In [1]:
import pandas as pd

# Using parse_dates to convert date columns from text to real datetime values, and specifying the date format with date_format
bge_da = pd.read_csv("../data/raw/bge_2025_da_hrl_lmps.csv",
                     parse_dates=["datetime_beginning_utc","datetime_beginning_ept"],
                     date_format={"datetime_beginning_utc": "%m/%d/%Y %I:%M:%S %p",
                                  "datetime_beginning_ept": "%m/%d/%Y %I:%M:%S %p"})

In [2]:
bge_da.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime_beginning_utc  8760 non-null   datetime64[us]
 1   datetime_beginning_ept  8760 non-null   datetime64[us]
 2   pnode_id                8760 non-null   int64         
 3   pnode_name              8760 non-null   str           
 4   voltage                 0 non-null      float64       
 5   equipment               0 non-null      float64       
 6   type                    8760 non-null   str           
 7   zone                    0 non-null      float64       
 8   system_energy_price_da  8760 non-null   float64       
 9   total_lmp_da            8760 non-null   float64       
 10  congestion_price_da     8760 non-null   float64       
 11  marginal_loss_price_da  8760 non-null   float64       
 12  row_is_current          8760 non-null   bool          
 13 

In [3]:
bge_da.describe()

,datetime_beginning_utc,datetime_beginning_ept,pnode_id,voltage,equipment,zone,system_energy_price_da,total_lmp_da,congestion_price_da,marginal_loss_price_da,version_nbr
count,8760,8760,8760.0,0.0,0.0,0.0,8760.000000,8760.000000,8760.000000,8760.000000,8760.0
mean,2025-07-02 16:30:00,2025-07-02 12:09:06.986301,51292.0,NaN,NaN,NaN,47.028871,57.373200,8.199609,2.144720,1.0
min,2025-01-01 05:00:00,2025-01-01 00:00:00,51292.0,NaN,NaN,NaN,11.710000,11.911439,-53.667279,-0.470398,1.0
25%,2025-04-02 10:45:00,2025-04-02 06:45:00,51292.0,NaN,NaN,NaN,30.340000,33.648900,0.599940,0.877276,1.0
50%,2025-07-02 16:30:00,2025-07-02 12:30:00,51292.0,NaN,NaN,NaN,38.735000,44.910950,3.757831,1.675140,1.0
75%,2025-10-01 22:15:00,2025-10-01 18:15:00,51292.0,NaN,NaN,NaN,53.070000,65.564132,10.835236,2.746591,1.0
max,2026-01-01 04:00:00,2025-12-31 23:00:00,51292.0,NaN,NaN,NaN,537.130000,609.740467,232.921752,30.717230,1.0
std,NaN,NaN,0.0,NaN,NaN,NaN,32.477654,45.099342,14.520829,2.158877,0.0


In [4]:
bge_da.pnode_name.value_counts()

pnode_name
BGE    8760
Name: count, dtype: int64

In [5]:
# checking that all values are current
bge_da.row_is_current.value_counts()

row_is_current
True    8760
Name: count, dtype: int64

In [6]:
bge_da.version_nbr.value_counts()

version_nbr
1    8760
Name: count, dtype: int64

In [7]:
#Checking that the data goes up to the last hour of the year
bge_da.datetime_beginning_ept.max()

Timestamp('2025-12-31 23:00:00')

In [8]:
# Making the UTC column timezone aware
bge_da["datetime_beginning_utc"] = bge_da["datetime_beginning_utc"].dt.tz_localize("UTC")

In [9]:
 #Setting the UTC column as the dataframe index
bge_da = bge_da.set_index("datetime_beginning_utc")

In [10]:
# Checking that the index column (UTC) is timezone aware
bge_da.index.dtype

datetime64[us, UTC]

In [11]:
bge_da.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8760 entries, 2025-01-01 05:00:00+00:00 to 2026-01-01 04:00:00+00:00
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime_beginning_ept  8760 non-null   datetime64[us]
 1   pnode_id                8760 non-null   int64         
 2   pnode_name              8760 non-null   str           
 3   voltage                 0 non-null      float64       
 4   equipment               0 non-null      float64       
 5   type                    8760 non-null   str           
 6   zone                    0 non-null      float64       
 7   system_energy_price_da  8760 non-null   float64       
 8   total_lmp_da            8760 non-null   float64       
 9   congestion_price_da     8760 non-null   float64       
 10  marginal_loss_price_da  8760 non-null   float64       
 11  row_is_current          8760 non-null   bool          
 12  version_nbr

Assert checks for `bge_da`: flag the notebook if the number of hours is wrong, if there are duplicate UTC timestamps, or if any price is missing.

In [12]:
assert len(bge_da) == 8760, f"Expected 8,760 hours, found {len(bge_da)}"

In [13]:
assert bge_da.index.is_unique, "Duplicate UTC timestamps found"

In [14]:
price_cols = ["system_energy_price_da", "total_lmp_da","congestion_price_da","marginal_loss_price_da"]
assert bge_da[price_cols].isna().sum().sum() == 0, "Null values found in price columns"

In [15]:
# Confirming that there are two duplicate Fall EPT timestamps for daylight savings 
bge_da.datetime_beginning_ept.duplicated(keep=False).sum()

np.int64(2)

In [16]:
bge_da.datetime_beginning_ept.value_counts(dropna=False)

datetime_beginning_ept
2025-11-02 01:00:00    2
2025-01-01 00:00:00    1
2025-01-01 01:00:00    1
2025-01-01 02:00:00    1
2025-01-01 03:00:00    1
                      ..
2025-12-31 19:00:00    1
2025-12-31 20:00:00    1
2025-12-31 21:00:00    1
2025-12-31 22:00:00    1
2025-12-31 23:00:00    1
Name: count, Length: 8759, dtype: int64

In [17]:
#Creating a variable DST_spring to visualize the missing EPT timestamp for Spring daylight savings
DST_spring = bge_da[bge_da["datetime_beginning_ept"].dt.date == pd.to_datetime("2025-03-09").date()]

In [18]:
DST_spring[["datetime_beginning_ept"]]

,datetime_beginning_ept
datetime_beginning_utc,
2025-03-09 05:00:00+00:00,2025-03-09 00:00:00
2025-03-09 06:00:00+00:00,2025-03-09 01:00:00
2025-03-09 07:00:00+00:00,2025-03-09 03:00:00
2025-03-09 08:00:00+00:00,2025-03-09 04:00:00
2025-03-09 09:00:00+00:00,2025-03-09 05:00:00
2025-03-09 10:00:00+00:00,2025-03-09 06:00:00
2025-03-09 11:00:00+00:00,2025-03-09 07:00:00
2025-03-09 12:00:00+00:00,2025-03-09 08:00:00
2025-03-09 13:00:00+00:00,2025-03-09 09:00:00


**Why UTC is the index:** Eastern time repeats one hour on Nov 2 (clocks fall back) and skips one on Mar 9 (clocks spring forward), so Eastern timestamps are not unique. UTC never repeats or skips, so it is the safe key for joining and checking rows; Eastern time is kept only for calendar grouping and labels.

### BGE real-time
Parse dates, confirm rows are current, make the UTC column timezone aware, set it as the index, and run the assert checks.

In [19]:
bge_rt = pd.read_csv("../data/raw/bge_2025_rt_hrl_lmps.csv",
                     parse_dates=["datetime_beginning_utc","datetime_beginning_ept"],
                     date_format={"datetime_beginning_utc": "%m/%d/%Y %I:%M:%S %p",
                                  "datetime_beginning_ept": "%m/%d/%Y %I:%M:%S %p"})

In [20]:
bge_rt.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime_beginning_utc  8760 non-null   datetime64[us]
 1   datetime_beginning_ept  8760 non-null   datetime64[us]
 2   pnode_id                8760 non-null   int64         
 3   pnode_name              8760 non-null   str           
 4   voltage                 0 non-null      float64       
 5   equipment               0 non-null      float64       
 6   type                    8760 non-null   str           
 7   zone                    0 non-null      float64       
 8   system_energy_price_rt  8760 non-null   float64       
 9   total_lmp_rt            8760 non-null   float64       
 10  congestion_price_rt     8760 non-null   float64       
 11  marginal_loss_price_rt  8760 non-null   float64       
 12  row_is_current          8760 non-null   bool          
 13 

In [21]:
bge_rt.describe()

,datetime_beginning_utc,datetime_beginning_ept,pnode_id,voltage,equipment,zone,system_energy_price_rt,total_lmp_rt,congestion_price_rt,marginal_loss_price_rt,version_nbr
count,8760,8760,8760.0,0.0,0.0,0.0,8760.000000,8760.000000,8760.000000,8760.000000,8760.0
mean,2025-07-02 16:30:00,2025-07-02 12:09:06.986301,51292.0,NaN,NaN,NaN,46.769244,56.501890,7.804247,1.928815,1.0
min,2025-01-01 05:00:00,2025-01-01 00:00:00,51292.0,NaN,NaN,NaN,1.100000,-8.609823,-204.504977,-1.542125,1.0
25%,2025-04-02 10:45:00,2025-04-02 06:45:00,51292.0,NaN,NaN,NaN,27.430000,29.280034,-0.262868,0.816134,1.0
50%,2025-07-02 16:30:00,2025-07-02 12:30:00,51292.0,NaN,NaN,NaN,35.980000,39.199124,0.818740,1.418036,1.0
75%,2025-10-01 22:15:00,2025-10-01 18:15:00,51292.0,NaN,NaN,NaN,51.545000,59.449369,7.022254,2.234760,1.0
max,2026-01-01 04:00:00,2025-12-31 23:00:00,51292.0,NaN,NaN,NaN,1837.950000,2021.929077,532.256389,102.722662,1.0
std,NaN,NaN,0.0,NaN,NaN,NaN,49.585078,68.252231,27.191924,2.589521,0.0


In [22]:
bge_rt.row_is_current.value_counts()

row_is_current
True    8760
Name: count, dtype: int64

In [23]:
# Making the UTC column timezone aware like the other dataframes
bge_rt["datetime_beginning_utc"] = bge_rt["datetime_beginning_utc"].dt.tz_localize("UTC")
bge_rt.datetime_beginning_utc.dtype

datetime64[us, UTC]

In [24]:
bge_rt = bge_rt.set_index("datetime_beginning_utc")

In [25]:
assert len(bge_rt) == 8760, f"Expected 8,760 hours, found {len(bge_rt)}"

In [26]:
assert bge_rt.index.is_unique, "Duplicate UTC timestamps found"

In [27]:
price_cols_rt = ["system_energy_price_rt", "total_lmp_rt", "congestion_price_rt", "marginal_loss_price_rt"]
assert bge_rt[price_cols_rt].isna().sum().sum() == 0, "Null values found in price columns"

### Western Hub day-ahead
Same steps as above.

In [28]:
wh_da = pd.read_csv("../data/raw/WH_2025_da_hrl_lmps.csv",
                     parse_dates=["datetime_beginning_utc","datetime_beginning_ept"],
                     date_format={"datetime_beginning_utc": "%m/%d/%Y %I:%M:%S %p",
                                  "datetime_beginning_ept": "%m/%d/%Y %I:%M:%S %p"})

In [29]:
wh_da.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   datetime_beginning_utc  8760 non-null   datetime64[us]
 1   datetime_beginning_ept  8760 non-null   datetime64[us]
 2   pnode_id                8760 non-null   int64         
 3   pnode_name              8760 non-null   str           
 4   voltage                 0 non-null      float64       
 5   equipment               0 non-null      float64       
 6   type                    8760 non-null   str           
 7   zone                    0 non-null      float64       
 8   system_energy_price_da  8760 non-null   float64       
 9   total_lmp_da            8760 non-null   float64       
 10  congestion_price_da     8760 non-null   float64       
 11  marginal_loss_price_da  8760 non-null   float64       
 12  row_is_current          8760 non-null   bool          
 13 

In [30]:
wh_da.row_is_current.value_counts()

row_is_current
True    8760
Name: count, dtype: int64

In [31]:
wh_da.datetime_beginning_utc.dtype

dtype('<M8[us]')

In [32]:
# Making UTC column timezone aware like the other dataframes
wh_da["datetime_beginning_utc"] = wh_da["datetime_beginning_utc"].dt.tz_localize("UTC")

In [33]:
# Confirming new data type
wh_da.datetime_beginning_utc.dtype

datetime64[us, UTC]

In [34]:
wh_da = wh_da.set_index("datetime_beginning_utc")

In [35]:
assert len(wh_da) == 8760, f"Expected 8,760 hours, found {len(wh_da)}"

In [36]:
assert wh_da.index.is_unique, "Duplicate UTC timestamps found"

In [37]:
price_cols = ["system_energy_price_da", "total_lmp_da","congestion_price_da","marginal_loss_price_da"]
assert wh_da[price_cols].isna().sum().sum() == 0, "Null values found in price columns"

### Western Hub real-time
Same steps as above.

In [38]:
wh_rt = pd.read_csv("../data/raw/WH_2025_rt_hrl_lmps.csv",
                     parse_dates=["datetime_beginning_utc","datetime_beginning_ept"],
                     date_format={"datetime_beginning_utc": "%m/%d/%Y %I:%M:%S %p",
                                  "datetime_beginning_ept": "%m/%d/%Y %I:%M:%S %p"})

In [39]:
wh_rt.row_is_current.value_counts()

row_is_current
True    8760
Name: count, dtype: int64

In [40]:
wh_rt.datetime_beginning_utc.dtype

dtype('<M8[us]')

In [41]:
# Making UTC column timezone aware like the other dataframes
wh_rt["datetime_beginning_utc"] = wh_rt["datetime_beginning_utc"].dt.tz_localize("UTC")

In [42]:
# Confirming new data type
wh_rt.datetime_beginning_utc.dtype

datetime64[us, UTC]

In [43]:
wh_rt = wh_rt.set_index("datetime_beginning_utc")

In [44]:
assert len(wh_rt) == 8760, f"Expected 8,760 hours, found {len(wh_rt)}"

In [45]:
assert wh_rt.index.is_unique, "Duplicate UTC timestamps found"

In [46]:
assert wh_rt[price_cols_rt].isna().sum().sum() == 0, "Null values found in price columns"

### Combine into one tidy table
Rename columns to common names so the day-ahead and real-time tables can be stacked.

In [47]:
# Creating a renaming dictionary for day ahead columns
rename_da = {
      "datetime_beginning_ept": "timestamp_ept",
      "total_lmp_da": "lmp",
      "system_energy_price_da": "energy",
      "congestion_price_da": "congestion",
      "marginal_loss_price_da": "loss",
  }
bge_da_tidy = bge_da.rename(columns=rename_da) 

In [48]:
# Renaming the index column
bge_da_tidy.index.name="timestamp_utc"

In [49]:
# Creating two new columns to identify data when tables are combined
bge_da_tidy["market"] = "DA"
bge_da_tidy["node"] = "BGE"

In [50]:
# Target columns, shared by all four tables
target_cols = ["timestamp_ept", "node", "market", "lmp", "energy", "congestion", "loss"]
bge_da_tidy = bge_da_tidy[target_cols]

In [51]:
bge_da_tidy.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8760 entries, 2025-01-01 05:00:00+00:00 to 2026-01-01 04:00:00+00:00
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp_ept  8760 non-null   datetime64[us]
 1   node           8760 non-null   str           
 2   market         8760 non-null   str           
 3   lmp            8760 non-null   float64       
 4   energy         8760 non-null   float64       
 5   congestion     8760 non-null   float64       
 6   loss           8760 non-null   float64       
dtypes: datetime64[us](1), float64(4), str(2)
memory usage: 592.4 KB


In [52]:
# Renaming existing columns, adding identifying columns, and selecting target columns for the Western Hub DA table
wh_da_tidy = wh_da.rename(columns=rename_da)
wh_da_tidy.index.name = "timestamp_utc"
wh_da_tidy["node"] = "WESTERN HUB"
wh_da_tidy["market"] = "DA"
wh_da_tidy = wh_da_tidy[target_cols]

In [53]:
# Creating a renaming dictionary for real time columns
rename_rt = {
      "datetime_beginning_ept": "timestamp_ept",
      "total_lmp_rt": "lmp",
      "system_energy_price_rt": "energy",
      "congestion_price_rt": "congestion",
      "marginal_loss_price_rt": "loss",
  }

In [54]:
# Renaming existing columns, adding identifying columns, and selecting target columns for the BGE RT table
bge_rt_tidy = bge_rt.rename(columns=rename_rt)
bge_rt_tidy.index.name = "timestamp_utc"
bge_rt_tidy["market"] = "RT"
bge_rt_tidy["node"] = "BGE"
bge_rt_tidy = bge_rt_tidy[target_cols]

Western Hub RT table:

In [55]:
wh_rt_tidy = wh_rt.rename(columns=rename_rt)
wh_rt_tidy.index.name = "timestamp_utc"
wh_rt_tidy["market"] = "RT"
wh_rt_tidy["node"] = "WESTERN HUB"
wh_rt_tidy = wh_rt_tidy[target_cols]

In [56]:
#Combining all pared down tables into a new table 
bge_wh_table = pd.concat([bge_da_tidy, bge_rt_tidy, wh_da_tidy, wh_rt_tidy])

In [57]:
bge_wh_table.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 35040 entries, 2025-01-01 05:00:00+00:00 to 2026-01-01 04:00:00+00:00
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp_ept  35040 non-null  datetime64[us]
 1   node           35040 non-null  str           
 2   market         35040 non-null  str           
 3   lmp            35040 non-null  float64       
 4   energy         35040 non-null  float64       
 5   congestion     35040 non-null  float64       
 6   loss           35040 non-null  float64       
dtypes: datetime64[us](1), float64(4), str(2)
memory usage: 2.4 MB


Assert checks for the combined table.

In [58]:
# Checking that the correct number of rows is there (4 series x 8,760 hours)
assert len(bge_wh_table) == 35040, f"Expected 35,040 rows, found {len(bge_wh_table)}"
assert isinstance(bge_wh_table.index, pd.DatetimeIndex) and str(bge_wh_table.index.tz) == "UTC", "Combined index is not a UTC DatetimeIndex"

In [59]:
# Checking the Energy + Congestion + Loss is always equal to LMP
diff = (bge_wh_table["energy"] + bge_wh_table["congestion"] + bge_wh_table["loss"]) - bge_wh_table["lmp"]
assert diff.abs().max() <= 0.01, "Energy, Congestion, and Loss do not equal LMP (off by more than $0.01)"

In [60]:
# Confirming the DA system energy price is the same at both nodes
bge_da_iso = bge_wh_table[(bge_wh_table["market"] == "DA") & (bge_wh_table["node"] == "BGE")]
wh_da_iso = bge_wh_table[(bge_wh_table["market"] == "DA") & (bge_wh_table["node"] == "WESTERN HUB")]
da_energy_diff = bge_da_iso["energy"] - wh_da_iso["energy"]
assert len(da_energy_diff) == 8760, "DA timestamps did not line up one-to-one"
assert da_energy_diff.abs().max() <= 0.01, "DA system energy price differs between BGE and Western Hub"

In [61]:
# Confirming the RT system energy price is the same at both nodes
bge_rt_iso = bge_wh_table[(bge_wh_table["market"] == "RT") & (bge_wh_table["node"] == "BGE")]
wh_rt_iso = bge_wh_table[(bge_wh_table["market"] == "RT") & (bge_wh_table["node"] == "WESTERN HUB")]
rt_energy_diff = bge_rt_iso["energy"] - wh_rt_iso["energy"]
assert len(rt_energy_diff) == 8760, "RT timestamps did not line up one-to-one"
assert rt_energy_diff.abs().max() <= 0.01, "RT system energy price differs between BGE and Western Hub"

In [62]:
bge_wh_table.to_parquet("../data/processed/lmp_2025.parquet")

In [63]:
check = pd.read_parquet("../data/processed/lmp_2025.parquet")
assert len(check) == 35040, f"Saved file has {len(check)} rows, expected 35,040"
check.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 35040 entries, 2025-01-01 05:00:00+00:00 to 2026-01-01 04:00:00+00:00
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp_ept  35040 non-null  datetime64[us]
 1   node           35040 non-null  str           
 2   market         35040 non-null  str           
 3   lmp            35040 non-null  float64       
 4   energy         35040 non-null  float64       
 5   congestion     35040 non-null  float64       
 6   loss           35040 non-null  float64       
dtypes: datetime64[us](1), float64(4), str(2)
memory usage: 2.4 MB
